In [1]:
include("trotter.jl")
using .Trotter
using Zygote
using LinearAlgebra

Lvec = (2, 3)
N = Int(prod(Lvec))
n_up, n_dn = (2, 2)
mb = TamFermion.fullSlaterMomBasis(Lvec, n_up, n_dn)

println("(n_up,n_dn)=($n_up,$n_dn): D = $(mb["ints"].size) Slater dets across $(mb["qtot_unique"].size) qtot blocks")
println("qtot block sizes {code: dim}:", Dict(zip(mb["qtot_unique"], mb["counts"])))

# isolate the qtot = 0 and qtot = 1 sectors (C-ravel codes; (0,0)->0, (0,1)->1 for a 3x2 BZ)
basis0 = mb["ints"][mb["qtot"] .== 0]
basis1 = mb["ints"][mb["qtot"] .== 1]
println("dim(qtot=0) = $(basis0.size), dim(qtot=1) = $(basis1.size))")

(n_up,n_dn)=(2,2): D = (225,) Slater dets across (6,) qtot blocks
qtot block sizes {code: dim}:Dict(0 => 39, 4 => 36, 5 => 36, 2 => 39, 3 => 36, 1 => 39)
dim(qtot=0) = (39,), dim(qtot=1) = (39,))


In [ ]:
using KrylovKit
H,_,_ = TamFermion.Hubbard(1, 8.273684210526316, Lvec, (n_up, n_dn))
eigsolve(H, rand(size(H, 1)), 1, :SR)

([-4.128023474226733, -3.5227663559731646, -2.062636388090084, -1.6175877191311325, -1.6175877191310584, 7.105427357601002e-15, 1.4579631407749751, 4.273684210526303, 4.273684210526323], Vector{ComplexF64}[[0.049272787847260247 + 0.0im, 2.8398915576106575e-16 + 0.0im, -0.12734177481438433 + 0.0im, 0.12734177481438308 + 0.0im, -5.234358606892055e-16 + 0.0im, 0.24678498193518658 + 0.0im, 2.8244085607562255e-16 + 0.0im, -0.04927278784726028 + 0.0im, -0.1273417748143841 + 0.0im, -0.12734177481438347 + 0.0im  …  -0.12734177481438377 + 0.0im, -0.12734177481438372 + 0.0im, -0.04927278784726083 + 0.0im, 3.914145505952494e-16 + 0.0im, 0.24678498193518664 + 0.0im, 2.288269295928353e-16 + 0.0im, -0.12734177481438383 + 0.0im, 0.12734177481438314 + 0.0im, -5.737485659953478e-16 + 0.0im, 0.049272787847260545 + 0.0im], [-1.1410454947845565e-16 + 0.0im, -0.08687582925179432 + 0.0im, -0.1281033031490557 + 0.0im, -0.12810330314905627 + 0.0im, -0.08687582925179504 + 0.0im, -1.3583376360705603e-15 + 0.0im

: 

([-10.61445941415064], Vector{ComplexF64}[[0.10145946319674698 + 0.0im, 3.248852395020002e-5 + 0.0im, -3.248852394951828e-5 + 0.0im, 0.1151096453228025 + 0.0im, 3.248852394956652e-5 + 0.0im, -0.1151096453228025 + 0.0im, 3.248852394986143e-5 + 0.0im, 3.866567725712288e-5 + 0.0im, 3.866567725778513e-5 + 0.0im, 0.13036553883596416 + 0.0im  …  0.13036553883596425 + 0.0im, -3.866567725764481e-5 + 0.0im, 3.86656772576345e-5 + 0.0im, -3.2488523949867255e-5 + 0.0im, -0.11510964532280177 + 0.0im, -3.2488523949848885e-5 + 0.0im, 0.11510964532280242 + 0.0im, 3.248852395013241e-5 + 0.0im, 3.248852394966932e-5 + 0.0im, 0.10145946319674709 + 0.0im]], ConvergenceInfo: one converged value after 3 iterations and 54 applications of the linear map;
norms of residuals are given by (4.41e-14).)

In [ ]:
Lvec = (2,3)
fgates = TamFermion.enumerate_ferm_excitations(2,Lvec; conserve_sz = true, conserve_mom=true, omit_hermitian_redundancy=false)
println("Num fgates: $(length(fgates)) \t expected count: $((3N^3+2N^2)/4)" )


Num fgates: 180 	 expected count: 180.0


In [3]:
p = 2 # order of excitation
gates = TamFermion.enumerate_ferm_excitations(p, Lvec; conserve_mom=true, conserve_sz=true)
println("enumerated $(length(gates)) gate labels (= 2 x #tau)" )

sig = 1
A = randn(length(gates)).*sig
ops = TamFermion.fgateToExpSector(gates, A, N, basis0)   # object array of LinearOperators, each (d0 x d0)
println("built $(length(ops)) LinearOperators on the qtot=0 sector (each $(basis0.size) x $(basis0.size))")

enumerated 228 gate labels (= 2 x #tau)
built 228 LinearOperators on the qtot=0 sector (each (71188,) x (71188,))


In [ ]:
d0 = basis0.size
v0 = zeros(ComplexF64, d0); v0[1] = 1.0                                       # reference Slater determinant
vt = randn(d0) .+ 1im*randn(d0); normalize!(vt)  # target state

function apply_U(ops, v)                                                              # U|v> = op_{K-1}...op_1 op_0 |v>
	for op in ops
        v = op * v
    end
	return v
end


println("|<vt|vt>|^2 = $(abs2(dot(vt, vt)))")
psi = apply_U(ops, v0)
fidelity = abs2(dot(vt, psi))                                              # |<vt|U|v0>|^2
println("\n||U v0|| = $(norm(psi))  (U unitary -> 1)")
println("fidelity |<vt|U|v0>|^2 = $(fidelity)  (small for random A; this is the objective you'd maximize over A)")
println("self-consistency |<psi|psi>|^2 = $(abs2(dot(psi, psi))) (should be 1)")

|<vt|vt>|^2 = 1.0

||U v0|| = 0.9999999999999997  (U unitary -> 1)
fidelity |<vt|U|v0>|^2 = 2.8163232569991234e-5  (small for random A; this is the objective you'd maximize over A)
self-consistency |<psi|psi>|^2 = 0.9999999999999989 (should be 1)


In [6]:
basis0 = mb["ints"][mb["qtot"] .== 0]
d0 = basis0.size

# reference and target state
v0 = zeros(ComplexF64, d0); v0[1] = 1.0                                       # reference Slater determinant
vt = randn(d0) .+ 1im*randn(d0); normalize!(vt)

# constructing gates
p = 2 # order of excitation
gates = TamFermion.enumerate_ferm_excitations(p, Lvec; conserve_mom=true, conserve_sz=true)

sig = 1
A = randn(length(gates)*2).*sig
tau_terms = TamFermion.fgateToTauSector(gates, N, basis0)

f(x) = TrotterOptimization.adjoint_loss(x, gates, tau_terms, v0, vt, basis0, N; num_exponentials=2)
@time gradient(f, A)

  0.896605 seconds (297.78 k allocations: 293.057 MiB, 65.66% gc time, 14.83% compilation time: 100% of which was recompilation)


([-0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0  …  -4.672504396692333e-5, -4.672504396692332e-5, 1.2901379228837048e-5, 1.290137922883704e-5, 4.5786729503597936e-5, 4.578672950359792e-5, 3.4020102104649635e-5, 3.402010210464962e-5, 0.00014749066998089765, 0.00014749066998089762],)

In [2]:
using Lattices, Combinatorics, SparseArrays, JLD2, HDF5, KrylovKit, Zygote
# try
#     using CUDA
# catch
# end
include("utility_functions.jl")
include("ed_objects.jl")
include("ed_functions.jl")
include("ed_optimization.jl")


In [ ]:
folder = "data/N=(4, 4)_3x3"
U_values, target_vecs, indexer, precomputed_structures, N_elec, spin_conserved, use_symmetry, sign_convention = load_ED_data(folder; verbose=true)

ref_u_idx = 1
u_idx = 25
state1 = target_vecs[ref_u_idx, :]
state2 = target_vecs[u_idx, :]

order = 2
num_exponentials = 1
operator_cache = Dict()
antihermitian = false
loss = 1 - abs2(state1' * state2)

struct_data = ensure_operator_structure!(order, operator_cache, indexer, spin_conserved, use_symmetry, false, sign_convention, precomputed_structures, antihermitian, loss * 100)

ops = struct_data[:ops]
rows = struct_data[:rows]
cols = struct_data[:cols]
signs = struct_data[:signs]
param_index_map = struct_data[:param_index_map]
parameter_mapping = struct_data[:parameter_mapping]
parity = struct_data[:parity]
dim = length(indexer.inv_comb_dict)

P = use_symmetry ? length(struct_data[:sym_data][1]) : length(struct_data[:t_keys])
t_vals = randn(P * num_exponentials) .* 0.01

println("Warming up CPU Adjoint Gradient...")
loss_val, back = Zygote.pullback(t -> adjoint_loss(t, ops, rows, cols, signs, param_index_map, parameter_mapping, parity, dim, state2, state1, nothing, !use_symmetry, antihermitian; num_exponentials=num_exponentials), t_vals)
grad = back(1.0)[1]

println("Benchmarking CPU Adjoint Gradient:")
@time begin
    loss_val, back = Zygote.pullback(t -> adjoint_loss(t, ops, rows, cols, signs, param_index_map, parameter_mapping, parity, dim, state2, state1, nothing, !use_symmetry, antihermitian; num_exponentials=num_exponentials), t_vals)
    grad = back(1.0)[1]
end

# if @isdefined(CUDA) && CUDA.functional()
#     println("\nWarming up GPU Adjoint Gradient...")
#     ops_gpu = ops
#     state1_gpu = CUDA.CuArray(state1)
#     state2_gpu = CUDA.CuArray(state2)
#     loss_val_gpu, back_gpu = Zygote.pullback(t -> gpu_adjoint_loss(t, ops_gpu, rows, cols, signs, param_index_map, parameter_mapping, parity, dim, state2_gpu, state1_gpu, nothing, !use_symmetry, antihermitian; num_exponentials=num_exponentials), t_vals)
#     grad_gpu = back_gpu(1.0)[1]

#     println("Benchmarking GPU Adjoint Gradient:")
#     @time begin
#         loss_val_gpu, back_gpu = Zygote.pullback(t -> gpu_adjoint_loss(t, ops_gpu, rows, cols, signs, param_index_map, parameter_mapping, parity, dim, state2_gpu, state1_gpu, nothing, !use_symmetry, antihermitian; num_exponentials=num_exponentials), t_vals)
#         grad_gpu = back_gpu(1.0)[1]
#     end
# end


Meta data:


Dict{String, Any} with 8 entries:
  "U_values"       => [1.0e-5, 0.001, 0.0013434, 0.00180472, 0.00242446, 0.0032…
  "maxiters"       => 200
  "optimizer"      => "BFGS"
  "sites"          => "3x3"
  "solver"         => "Lanczos"
  "basis"          => "adiabatic"
  "bc"             => "periodic"
  "electron count" => (4, 4)

Finding best energy sector
U=1.0e-5 k=[1, 9] [-13.999983333347735, -13.999982222235369]
U=0.001 k=[1, 5] [-13.998333477364922, -13.998222353909183]
U=0.0013433993325989001 k=[1, 5] [-13.99776126104841, -13.997611972177973]
U=0.001804721766827172 k=[1, 8] [-13.996992599498883, -13.996792034654373]
U=0.0024244620170823282 k=[1, 9] [-13.99596007658007, -13.995690619359008]
U=0.0032570206556597828 k=[1, 9] [-13.99457316012148, -13.994211138010073]
U=0.004375479375074184 k=[1, 6] [-13.992710291745759, -13.99222389110663]
U=0.0058780160722749115 k=[1, 9] [-13.990208282762879, -13.98955474353184]
U=0.007896522868499725 k=[1, 6] [-13.98684810906754, -13.985969948361666]
U=0.010608183551394472 k=[1, 6] [-13.982335900994949, -13.981155825940668]
U=0.014251026703029992 k=[1, 8] [-13.976277536783462, -13.974691585247985]
U=0.019144819761699575 k=[1, 6] [-13.96814474901065, -13.966013029556516]
U=0.025719138090593455 k=[1, 6] [-13.957230020534624, -13.954364190976056]
U=0.03455107294592222 k=[1, 8]

585-element Vector{Float64}:
  0.0005474728127403261
  0.015384988667460756
 -0.01564151113316967
 -0.015918058334902965
  0.016047339910179374
  0.026171332189287713
  0.006217888838621024
  0.025629026719430185
 -0.015924572745976893
  0.0827355332391735
 -0.015656741993184346
  0.00246644564416302
  0.008720862383308738
  ⋮
 -0.00011786990277789632
  1.162872761654237e-6
  0.00011998668658875421
  0.00020076611046760624
 -0.0004455568608200554
  0.00012765734268961788
 -0.0005862679494307563
  3.602618348612091e-5
  0.0002038642018148851
 -2.6929154366247943e-5
 -6.01051780712048e-6
 -0.0007338838160438683

: 